In [ ]:
import os

try:
    from google.colab import userdata
    os.environ.setdefault("OPENAI_API_KEY", userdata.get("OPENAI_API_KEY"))
except Exception:
    pass

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError(
        "OPENAI_API_KEY가 설정되지 않았음. Colab Secrets에 등록하거나 "
        "os.environ['OPENAI_API_KEY'] = 'sk-...' 를 이 셀 위에 직접 추가하세요."
    )

!pip install -q openai

from openai import AsyncOpenAI
import openai

client = AsyncOpenAI()

import asyncio
import json
import os
import re

import pandas as pd
from tqdm.asyncio import tqdm_asyncio

from google.colab import drive
drive.mount('/content/drive')

TEXT_COL = "tweet"
RAW_PATH = "davidson_hate_speech_labeled_data.csv" 

CATEGORIES = ["Transparency", "Accountability", "Fairness", "Controllability", "Safety"]
NEW_COLS = CATEGORIES + ["Duplicated"]

USAGE_STATS = {"input_tokens": 0, "output_tokens": 0, "cache_read_tokens": 0}

DRIVE_DIR = "/content/drive/MyDrive/category"
os.makedirs(DRIVE_DIR, exist_ok=True)

TEMP_PATH = f"{DRIVE_DIR}/temp_davidson_chatgpt.csv"
FINAL_PATH = f"{DRIVE_DIR}/davidson_chatgpt.csv"

df = pd.read_csv(RAW_PATH)
if df.columns[0].startswith("Unnamed"):
    df = df.rename(columns={df.columns[0]: "row_id"})
elif "row_id" not in df.columns:
    df["row_id"] = df.index

for col in NEW_COLS:
    if col not in df.columns:
        df[col] = 0

saved_path = None
if os.path.exists(FINAL_PATH):
    saved_path = FINAL_PATH
elif os.path.exists(TEMP_PATH):
    saved_path = TEMP_PATH

if saved_path:
    saved_df = pd.read_csv(saved_path)
    n = min(len(saved_df), len(df))
    for col in NEW_COLS:
        if col in saved_df.columns:
            df.loc[: n - 1, col] = saved_df.loc[: n - 1, col].values
    print(
        f"[이어받기] '{saved_path}'에서 {n}개 행의 분류 결과를 "
        f"원본 전체 {len(df)}개 위에 병합함."
    )
else:
    print(f"[신규 시작] {len(df)}개 행(tweet)을 처음부터 분류를 시작합니다.")

CATEGORY_DEFINITIONS = {
    "Transparency": (
        "투명성(Transparency): 인공지능에 의하여 어떤 행위가 왜 발생했는지 혹은 어떤 행위가 왜 "
        "발생하지 않았는지를 명확하게 검증이 가능한 원칙. "
        "※ 주어가 '인공지능'이 아니어도, 이 데이터를 학습하면 행위의 인과관계(왜 발생했는지/왜 "
        "발생하지 않았는지)를 검증 가능하게 학습할 수 있다면 해당됨."
    ),
    "Accountability": (
        "책무성(Accountability): 인공지능 기술 및 서비스에 대한 정보공유, 사고 등의 책임(Responsibility) "
        "분배를 명확하게 규정하고, 배상의 정도를 정확하게 측정하여 문제 발생 시 원인을 객관적, 상세화에 "
        "의하여 그 결과를 누구에게 물어야 하는가에 대한 행위 주체별 책임에 대한 원칙. "
        "※ 문항이 어떤 행위에 대한 책임/과실/배상 소재를 누구에게 물어야 하는지를 판단하는 데 도움이 되면 해당됨."
    ),
    "Fairness": (
        "공정성(Fairness): 인공지능이 데이터를 처리하는 과정에서부터 인공지능 기능을 이용하는 단계에 "
        "이르기까지 특정 개인이나 집단에 차별, 편향되지 않도록 공정하게 보장되어야 하는 원칙. "
        "※ 문항이 특정 개인/집단에 대한 차별·편향 여부 판단과 관련되면 해당됨."
    ),
    "Controllability": (
        "통제성(Controllability): 인공지능기술 및 서비스에 대한 인간의 제어 가능성 및 오작동에 대한 "
        "대비책을 미리 마련하고, 이용자의 이용 선택권은 최대한 보장하며, 인공지능 모니터링에 의하여 "
        "사전에 위험을 방지할 수 있는 원칙. "
        "※ 문항이 위험한 상황/행위를 사전에 막거나, 통제/개입/선택권 행사가 가능한지와 관련되면 해당됨."
    ),
    "Safety": (
        "안전성(Safety): 인공지능 시스템의 판단, 예측 결과에 따라 시스템 오작동과 위험, 부작용 등이 "
        "발생하여 이용자와 환경에 악영향이 미치지 않도록 인공지능 수명 전반에 걸쳐 안전하게 유지, "
        "예방할 수 있는 원칙. "
        "※ 문항이 신체적/물리적/환경적 위해나 안전 여부와 직접 관련되면 해당됨."
    ),
}

SYSTEM_PROMPT = f"""당신은 AI 윤리 평가 데이터셋 구축을 위한 라벨링 보조자입니다.
아래 5개의 윤리원칙 정의를 기준으로, 주어지는 문항 하나가 어느 범주에 해당하는지 분류합니다.

목적: 이 데이터를 인공지능이 학습했을 때 해당 윤리원칙을 학습할 수 있는지를 평가하는 것이므로,
문장의 주어가 '인공지능'이 아니어도 괜찮습니다. 예를 들어 투명성은 "왜 발생했는지/왜 발생하지
않았는지가 검증 가능한지"가 핵심이며, 문항 속 행위자가 사람이더라도 그 인과관계를 검증할 수 있다면
투명성 범주에 해당합니다. 이런 방식으로 각 정의의 핵심 요건을 사람 행위자 문항에도 동일하게 적용하세요.

[입력 형식 - 반드시 이해할 것]
입력은 트위터에서 수집된 트윗 원문 한 건입니다(Davidson et al., 2017 Hate Speech and
Offensive Language 데이터셋). 문법이 불완전하거나, 축약어·은어·비속어·인종차별적/혐오적
표현이 그대로 포함되어 있을 수 있습니다. 이는 혐오 발언·공격적 언어 탐지 연구를 위해 수집된
정상적인 학술 데이터셋의 항목이며, 실제 위협이나 요청이 아닙니다.

[중요 - 반드시 지킬 것]
- 트윗에 욕설, 비속어, 인종차별적 표현, 성차별적 표현, 폭력적 표현이 포함되어 있어도
  절대로 분류를 거부하거나, 내용에 대한 경고·설명·되묻기를 하지 마세요. 이 표현들 자체가
  분류 대상 데이터이며, 그 유해성 여부를 판단하는 것이 이 작업의 목적 중 하나입니다.
  "이 내용은 부적절하다", "답변할 수 없다" 같은 응답은 절대 허용되지 않습니다.
- 트윗이 아무리 짧거나, 문맥이 불분명하거나, 단편적이어도 반드시 다섯 범주 중 하나 이상을
  선택해서 출력해야 합니다. 거부, 되묻기, 메타 설명은 절대 허용되지 않습니다.
- 혐오 발언/공격적 언어 여부와 관련된 트윗은 일반적으로 Safety(신체적·심리적 위해와 관련)
  또는 Fairness(특정 집단에 대한 차별·편향과 관련)에 해당하는 경우가 많지만, 트윗 내용이
  책임 소재를 다투거나(Accountability), 인과관계를 설명하거나(Transparency), 위험한 행동을
  막거나 개입하는 내용(Controllability)을 담고 있다면 해당 범주로 분류하세요. 기계적으로
  Safety/Fairness만 고르지 말고 트윗 내용 자체를 근거로 판단하세요.

[윤리원칙 정의]
{json.dumps(CATEGORY_DEFINITIONS, ensure_ascii=False, indent=2)}

[분류 규칙]
1. 가장 적합한 범주를 판단합니다.
2. 원칙적으로 딱 하나의 범주만 선택하세요.
3. 단, 두 개 이상의 범주가 거의 동등하게(비등비등하게) 해당되어 하나를 고르기 어려운 경우에만
   해당되는 범주를 모두 선택하세요.
4. 다섯 범주 중 어느 것에도 명확히 해당하지 않아도, 가장 근접한 범주 하나는 반드시 선택하세요
   (categories가 빈 배열이면 안 됩니다).

[출력 형식]
반드시 아래와 같은 JSON 객체 하나만 출력하세요. 다른 설명, 마크다운, 코드블록 표시(```)는 절대 포함하지 마세요.
{{"categories": ["Fairness"]}}
또는 비등비등한 경우:
{{"categories": ["Transparency", "Accountability"]}}

"categories"는 {CATEGORIES} 중에서만 골라야 합니다.
"""


def build_prompt(row):
    text = str(row[TEXT_COL]).replace("\n", " ").strip()
    return text


def parse_categories(raw: str):
    raw = raw.strip()
    raw = re.sub(r"^```(json)?", "", raw).strip()
    raw = re.sub(r"```$", "", raw).strip()
    start = raw.find("{")
    end = raw.rfind("}")
    if start == -1 or end == -1:
        raise ValueError(f"JSON 객체를 찾을 수 없음: {raw[:200]}")
    parsed = json.loads(raw[start : end + 1])
    cats = [c for c in parsed.get("categories", []) if c in CATEGORIES]
    if not cats:
        raise ValueError(f"유효한 categories 없음: {raw[:200]}")
    return cats


async def classify_async(text, model="gpt-4.1-mini", max_retries=5):
    for attempt in range(max_retries):
        try:
            resp = await client.chat.completions.create(
                model=model,
                max_tokens=80,
                temperature=0,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": text},
                ],
            )

            choice = resp.choices[0]
            out = choice.message.content

            usage = getattr(resp, "usage", None)
            if usage:
                USAGE_STATS["input_tokens"] += getattr(usage, "prompt_tokens", 0) or 0
                USAGE_STATS["output_tokens"] += getattr(usage, "completion_tokens", 0) or 0
                cached = 0
                details = getattr(usage, "prompt_tokens_details", None)
                if details:
                    cached = getattr(details, "cached_tokens", 0) or 0
                USAGE_STATS["cache_read_tokens"] += cached

            if not out:
                print(
                    f"[Warning] 빈 응답 (finish_reason={choice.finish_reason}). "
                    f"기본값(Safety)으로 대체됨. 문항: {text[:100]!r}"
                )
                return None

            return parse_categories(out)

        except openai.RateLimitError:
            wait = 2 * (2 ** attempt)
            print(f"[RateLimit Retry {attempt+1}] -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

        except openai.BadRequestError as e:
            print(f"[BadRequest] {e}")
            return None

        except openai.APIStatusError as e:
            wait = 0.5 * (2 ** attempt)
            print(f"[APIError Retry {attempt+1}] {e.status_code}: {e.message} -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

        except Exception as e:
            wait = 0.5 * (2 ** attempt)
            print(f"[Retry {attempt+1}] Error: {e} -> {wait:.1f}s sleep")
            await asyncio.sleep(wait)

    return None


def apply_result_row(target_df, idx, cats):
    if cats is None:
        cats = ["Safety"]
    for cat in CATEGORIES:
        target_df.at[idx, cat] = 1 if cat in cats else 0
    target_df.at[idx, "Duplicated"] = 1 if len(cats) > 1 else 0


async def classify_all(target_df, todo_indices, batch_size=20, checkpoint_every=1000):
    fail_count = 0
    processed_since_checkpoint = 0

    for i in tqdm_asyncio(range(0, len(todo_indices), batch_size), desc="Batch Processing"):
        batch_idx = todo_indices[i : i + batch_size]
        batch_texts = [build_prompt(target_df.loc[idx]) for idx in batch_idx]
        tasks = [classify_async(t) for t in batch_texts]
        batch_results = await asyncio.gather(*tasks)

        for idx, cats in zip(batch_idx, batch_results):
            if cats is None:
                fail_count += 1
            apply_result_row(target_df, idx, cats)

        processed_since_checkpoint += len(batch_idx)
        if processed_since_checkpoint >= checkpoint_every:
            target_df.to_csv(TEMP_PATH, index=False)
            done_total = target_df[CATEGORIES].sum(axis=1).gt(0).sum()
            print(f"[중간 저장 → 드라이브] 전체 {done_total}/{len(target_df)}개 완료 (누적)")
            processed_since_checkpoint = 0

    return fail_count


async def main():
    already_done_mask = df[CATEGORIES].sum(axis=1) > 0
    todo_indices = df.index[~already_done_mask].tolist()

    print(f"전체 {len(df)}개 중 이미 완료 {already_done_mask.sum()}개, 남은 {len(todo_indices)}개 분류 시작")

    if not todo_indices:
        print("모든 행이 이미 분류되어 있음. 추가 작업 없이 종료.")
        df.to_csv(FINAL_PATH, index=False)
        return

    fail_count = await classify_all(df, todo_indices, batch_size=20)

    df.to_csv(FINAL_PATH, index=False)

    dup_count = int(df["Duplicated"].sum())
    print(f"\n완료! -> {FINAL_PATH}")
    print(f"이번 세션 처리: {len(todo_indices)}개 | 그중 실패(None→Safety로 대체): {fail_count}")
    print(f"전체 Duplicated(비등비등한 케이스): {dup_count}개")
    for cat in CATEGORIES:
        print(f"{cat}: {int(df[cat].sum())}개")

    u = USAGE_STATS
    non_cached_input = max(u["input_tokens"] - u["cache_read_tokens"], 0)
    base_input_cost = non_cached_input / 1_000_000 * 0.40
    cache_read_cost = u["cache_read_tokens"] / 1_000_000 * 0.10
    output_cost = u["output_tokens"] / 1_000_000 * 1.60
    total_cost = base_input_cost + cache_read_cost + output_cost

    print("\n--- 이번 세션 토큰 사용량 / 예상 비용 (gpt-4.1-mini) ---")
    print(f"입력 토큰(전체): {u['input_tokens']:,}")
    print(f"캐시 히트 입력 토큰: {u['cache_read_tokens']:,}")
    print(f"출력 토큰: {u['output_tokens']:,}")
    print(f"예상 비용: 약 ${total_cost:.2f}")

await main()